# Hidden Markov Model with Gaussian observation distributions

In [2]:
!pip install numpy
import numpy as np

class GaussianHMM:

    def __init__(
        self,
        initial_probs,
        transition_matrix,
        observation_means,
        observation_stds,
        num_steps=7,
        seed=None,
    ):
        self.initial_probs = np.asarray(initial_probs, dtype=float)
        self.transition_matrix = np.asarray(
            transition_matrix, dtype=float
        )
        self.observation_means = np.asarray(
            observation_means, dtype=float
        )
        self.observation_stds = np.asarray(
            observation_stds, dtype=float
        )
        self.num_steps = num_steps
        self.rng = np.random.default_rng(seed)

        self._validate_parameters()

    def _validate_parameters(self):
        n = len(self.initial_probs)

        if n == 0:
            raise ValueError("At least one hidden state is required.")

        if self.transition_matrix.shape != (n, n):
            raise ValueError("Transition matrix must have shape (n, n).")

        if (
            self.observation_means.shape != (n,)
            or self.observation_stds.shape != (n,)
        ):
            raise ValueError("Observation parameters must have length n.")

        if not np.all(np.isfinite(self.initial_probs)):
            raise ValueError("Initial probabilities must be finite.")

        if not np.all(np.isfinite(self.transition_matrix)):
            raise ValueError("Transition probabilities must be finite.")

        if np.any(self.initial_probs < 0):
            raise ValueError("Initial probabilities cannot be negative.")

        if np.any(self.transition_matrix < 0):
            raise ValueError("Transition probabilities cannot be negative.")

        if not np.isclose(self.initial_probs.sum(), 1.0):
            raise ValueError("Initial probabilities must sum to 1.")

        if not np.allclose(self.transition_matrix.sum(axis=1), 1.0):
            raise ValueError("Each transition row must sum to 1.")

        if not np.all(np.isfinite(self.observation_means)):
            raise ValueError("Observation means must be finite.")

        if (
            not np.all(np.isfinite(self.observation_stds))
            or np.any(self.observation_stds <= 0)
        ):
            raise ValueError("Observation standard deviations must be positive.")

        if not isinstance(self.num_steps, (int, np.integer)) or self.num_steps < 1:
            raise ValueError("num_steps must be a positive integer.")

    @property
    def num_states(self):
        return len(self.initial_probs)

    def state_probabilities(self, num_steps=None):
        """Return hidden-state probabilities for each time step."""
        steps = self.num_steps if num_steps is None else num_steps

        if not isinstance(steps, (int, np.integer)) or steps < 1:
            raise ValueError("num_steps must be a positive integer.")

        probabilities = np.zeros((steps, self.num_states))
        probabilities[0] = self.initial_probs

        for t in range(1, steps):
            probabilities[t] = (
                probabilities[t - 1] @ self.transition_matrix
            )

        return probabilities

    def mean(self, num_steps=None):
        """Expected observation at each time step."""
        probabilities = self.state_probabilities(num_steps)
        return probabilities @ self.observation_means

    def observation_variance(self, num_steps=None):
        """Variance of the observation at each time step."""
        probabilities = self.state_probabilities(num_steps)

        second_moment = probabilities @ (
            self.observation_stds**2 + self.observation_means**2
        )

        expected_mean = probabilities @ self.observation_means

        return second_moment - expected_mean**2

    def sample(self, num_steps=None):
        """Generate hidden states and corresponding observations."""
        steps = self.num_steps if num_steps is None else num_steps

        if not isinstance(steps, (int, np.integer)) or steps < 1:
            raise ValueError("num_steps must be a positive integer.")

        states = np.zeros(steps, dtype=int)
        observations = np.zeros(steps, dtype=float)

        states[0] = self.rng.choice(
            self.num_states, p=self.initial_probs
        )

        for t in range(1, steps):
            states[t] = self.rng.choice(
                self.num_states,
                p=self.transition_matrix[states[t - 1]],
            )

        observations = self.rng.normal(
            loc=self.observation_means[states],
            scale=self.observation_stds[states],
        )

        return states, observations

    def _log_emission_probabilities(self, observations):
        """Log probability density for every observation and state."""
        observations = np.asarray(observations, dtype=float)

        if observations.ndim != 1 or observations.size == 0:
            raise ValueError("Observations must be a non-empty 1D array.")

        if not np.all(np.isfinite(observations)):
            raise ValueError("Observations must be finite.")

        z = (
            observations[:, None] - self.observation_means[None, :]
        ) / self.observation_stds[None, :]

        return (
            -0.5 * np.log(2.0 * np.pi)
            - np.log(self.observation_stds)[None, :]
            - 0.5 * z**2
        )

    @staticmethod
    def _logsumexp(values):
        """Numerically stable log(sum(exp(values)))."""
        maximum = np.max(values, axis=0)
        return maximum + np.log(
            np.sum(np.exp(values - maximum), axis=0)
        )

    def log_likelihood(self, observations):
        """Log likelihood of an observed sequence using forward recursion."""
        log_emissions = self._log_emission_probabilities(observations)

        log_initial = np.full(self.num_states, -np.inf)
        positive = self.initial_probs > 0
        log_initial[positive] = np.log(self.initial_probs[positive])

        log_transition = np.full_like(self.transition_matrix, -np.inf)
        positive = self.transition_matrix > 0
        log_transition[positive] = np.log(
            self.transition_matrix[positive]
        )

        alpha = log_initial + log_emissions[0]

        for t in range(1, len(log_emissions)):
            alpha = log_emissions[t] + self._logsumexp(
                alpha[:, None] + log_transition
            )

        return float(self._logsumexp(alpha))

    def likelihood(self, observations):
        """Probability density of an observed sequence."""
        return float(np.exp(self.log_likelihood(observations)))

    def predict_states(self, observations):
        """Most likely hidden-state sequence using Viterbi decoding."""
        log_emissions = self._log_emission_probabilities(observations)

        log_initial = np.full(self.num_states, -np.inf)
        positive = self.initial_probs > 0
        log_initial[positive] = np.log(self.initial_probs[positive])

        log_transition = np.full_like(self.transition_matrix, -np.inf)
        positive = self.transition_matrix > 0
        log_transition[positive] = np.log(
            self.transition_matrix[positive]
        )

        scores = log_initial + log_emissions[0]
        backpointers = np.zeros(
            (len(log_emissions), self.num_states), dtype=int
        )

        for t in range(1, len(log_emissions)):
            candidates = scores[:, None] + log_transition

            backpointers[t] = np.argmax(candidates, axis=0)

            scores = (
                np.max(candidates, axis=0) + log_emissions[t]
            )

        states = np.zeros(len(log_emissions), dtype=int)
        states[-1] = int(np.argmax(scores))

        for t in range(len(states) - 2, -1, -1):
            states[t] = backpointers[t + 1, states[t + 1]]

        return states

## Define the original Hidden Markov Model

In [3]:
initial_probs = [0.2, 0.8]

transition_matrix = [
    [0.5, 0.5],
    [0.2, 0.8],
]

observation_means = [0.0, 15.0]
observation_stds = [5.0, 10.0]

model = GaussianHMM(
    initial_probs=initial_probs,
    transition_matrix=transition_matrix,
    observation_means=observation_means,
    observation_stds=observation_stds,
    num_steps=7,
    seed=42,
)

# 1. Calculate expected observations
print("Expected observation means:")
print(model.mean())

# 2. Calculate observation variances
print("\nObservation variances:")
print(model.observation_variance())

# 3. Generate a sample sequence
states, observations = model.sample()

print("\nGenerated hidden states:")
print(states)

print("\nGenerated observations:")
print(observations)

# 4. Evaluate the likelihood of the generated observations
print("\nLog likelihood:")
print(model.log_likelihood(observations))

print("\nLikelihood:")
print(model.likelihood(observations))

# 5. Infer the most likely hidden states from observations
predicted_states = model.predict_states(observations)

print("\nPredicted hidden states:")
print(predicted_states)

Expected observation means:
[12.       11.1      10.83     10.749    10.7247   10.71741  10.715223]

Observation variances:
[121.         123.79       124.3111     124.438999   124.47480991
 124.48532289 124.48845606]

Generated hidden states:
[1 1 1 1 0 1 1]

Generated observations:
[11.83757408 14.83198842  6.46956072 23.79397975  3.88895968 15.66030698
 26.27241207]

Log likelihood:
-25.603710476386826

Likelihood:
7.593636846381137e-12

Predicted hidden states:
[1 1 1 1 1 1 1]
